## Import

In [1]:
import pandas as pd

## Data

In [ ]:
orders = pd.read_csv('Cleaned_data/cleaned_orders.csv', parse_dates=['order_date'])
products = pd.read_csv('Cleaned_data/cleaned_products.csv')
returns = pd.read_csv('Cleaned_data/cleaned_returns.csv')
order_items = pd.read_csv('Cleaned_data/cleaned_order_items.csv', dtype={'promo_id_2': str})
customers = pd.read_csv('Cleaned_data/cleaned_customers.csv')
payments = pd.read_csv('Cleaned_data/cleaned_payments.csv')
web_traffic = pd.read_csv('Cleaned_data/cleaned_web_traffic.csv')
geography = pd.read_csv('Cleaned_data/cleaned_geography.csv')

## Answers

### **Q1.** Trong số các khách hàng có nhiều hơn một đơn hàng, **trung vị số ngày giữa hai lần mua liên tiếp** (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ *orders.csv*)

In [16]:
# Q1. Trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap)
orders_sorted = orders.sort_values(by=['customer_id', 'order_date'])
orders_sorted['prev_date'] = orders_sorted.groupby('customer_id')['order_date'].shift(1)
q1_ans = (orders_sorted['order_date'] - orders_sorted['prev_date']).dt.days.dropna().median()
print(f"Q1: Trung vị khoảng cách mua hàng: {q1_ans} ngày")

Q1: Trung vị khoảng cách mua hàng: 144.0 ngày


### **Q2.** Phân khúc sản phẩm (*segment*) nào trong products.csv có **tỷ suất lợi nhuận gộp trung bình** cao nhất, với công thức *(price − cogs)/price*?

In [32]:
# Q2. Phân khúc (segment) có tỷ suất lợi nhuận gộp trung bình cao nhất
products['margin'] = (products['price'] - products['cogs']) / products['price']
q2_ans = products.groupby('segment')['margin'].mean().idxmax()
print(f"Q2: Phân khúc sản phẩm có tỷ suất lợi nhuận gộp trung bình cao nhất: {q2_ans}")

Q2: Phân khúc sản phẩm có tỷ suất lợi nhuận gộp trung bình cao nhất: standard


### **Q3.** Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục **Streetwear** (join
*returns* với *products* theo *product_id*), **lý do trả hàng** nào xuất hiện nhiều nhất?

In [19]:
# Q3. Lý do trả hàng phổ biến nhất cho sản phẩm thuộc danh mục Streetwear
return_product = pd.merge(returns, products, on='product_id')
q3_ans = return_product[return_product['category'] == 'streetwear']['return_reason'].value_counts().idxmax()
print(f"Q3: Lý do trả hàng phổ biến nhất của Streetwear: {q3_ans}")

Q3: Lý do trả hàng phổ biến nhất của Streetwear: wrong_size


### **Q4.** Trong *web_traffic.csv*, nguồn truy cập (*traffic_source*) nào có **tỷ lệ thoát trung bình** (*bounce_rate*) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột *traffic_source*?

In [30]:
# Q4. Nguồn truy cập có tỷ lệ thoát (bounce_rate) trung bình thấp nhất
q4_ans = web_traffic.groupby('traffic_source')['bounce_rate'].mean().idxmin()
print(f"Q4: Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất trên tất cả các ngày xuất hiện nguồn đó: {q4_ans}")

Q4: Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất trên tất cả các ngày xuất hiện nguồn đó: email_campaign


### **Q5.** Tỷ lệ phần trăm các dòng trong *order_items.csv* có áp dụng khuyến mãi (tức là *promo_id* không *null*) xấp xỉ là bao nhiêu?

In [22]:
# Q5. Tỷ lệ phần trăm order_items có áp dụng khuyến mãi (promo_id không null)
promotion_ratio = order_items['promo_id'].notna().mean() * 100
print(f"Q5: Tỷ lệ dòng order_items có áp dụng khuyến mãi: {promotion_ratio:.2f}%")

Q5: Tỷ lệ dòng order_items có áp dụng khuyến mãi: 38.66%


### **Q6.** Trong *customers.csv*, xét các khách hàng có *age_group* khác *null*, nhóm tuổi nào có **số đơn hàng trung bình trên mỗi khách hàng** cao nhất? (tổng số đơn / số khách hàng trong nhóm)

In [23]:
# Q6. Nhóm tuổi có số đơn hàng trung bình / khách hàng cao nhất
valid_customers = customers.dropna(subset=['age_group'])
cust_orders = pd.merge(valid_customers, orders, on='customer_id')
# (Số lượng đơn hàng độc nhất) / (Số lượng khách hàng độc nhất trong nhóm)
orders_per_age = cust_orders.groupby('age_group')['order_id'].nunique()
cust_per_age = valid_customers.groupby('age_group')['customer_id'].nunique()
q6_ans = (orders_per_age / cust_per_age).idxmax()
print(f"Q6: Nhóm tuổi có số đơn trung bình cao nhất: {q6_ans}")

Q6: Nhóm tuổi có số đơn trung bình cao nhất: 55+


### **Q7.** Vùng (*region*) nào trong *geography.csv* tạo ra **tổng doanh thu cao nhất** trong *sales_train.csv*?

In [24]:
# Q7. Vùng (region) tạo ra tổng doanh thu cao nhất
# Doanh thu từng item = quantity * unit_price
order_items['revenue'] = order_items['quantity'] * order_items['unit_price']
orders_geo = pd.merge(orders, geography, on='zip', how='inner')
item_orders_geo = pd.merge(order_items, orders_geo, on='order_id', how='inner')
q7_ans = item_orders_geo.groupby('region')['revenue'].sum().idxmax()
print(f"Q7: Vùng tạo doanh thu cao nhất trong sales_train.csv: {q7_ans}")

Q7: Vùng tạo doanh thu cao nhất trong sales_train.csv: east


### **Q8.** Trong các đơn hàng có *order_status = ’cancelled’* trong *orders.csv*, **phương thức thanh toán** nào được sử dụng nhiều nhất?

In [25]:
# Q8. Phương thức thanh toán dùng nhiều nhất cho đơn hàng bị 'cancelled'
q8_ans = orders[orders['order_status'] == 'cancelled']['payment_method'].value_counts().idxmax()
print(f"Q8: Phương thức thanh toán cho đơn huỷ nhiều nhất: {q8_ans}")

Q8: Phương thức thanh toán cho đơn huỷ nhiều nhất: credit_card


### **Q9.** Trong bốn kích thước sản phẩm *(S, M, L, XL)*, kích thước nào có **tỷ lệ trả hàng cao nhất**, được định nghĩa là số bản ghi trong *returns* chia cho số dòng trong *order_items* (*join* với *products* theo *product_id*)?

In [26]:
# Q9. Kích cỡ sản phẩm có tỷ lệ trả hàng cao nhất
items_prod = pd.merge(order_items, products, on='product_id')
return_rates = {}
for size in ['S', 'M', 'L', 'XL']:
    num_ret = len(return_product[return_product['size'] == size])  # ret_prod đã tính ở Q3
    num_items = len(items_prod[items_prod['size'] == size])
    return_rates[size] = num_ret / num_items if num_items > 0 else 0

q9_ans = max(return_rates, key=return_rates.get)
print(f"Q9: Kích cỡ có tỷ lệ trả hàng cao nhất: {q9_ans}")

Q9: Kích cỡ có tỷ lệ trả hàng cao nhất: S


### **Q10.** Trong *payments.csv*, **kế hoạch trả góp** nào có **giá trị thanh toán trung bình trên mỗi đơn hàng** cao nhất?

In [28]:
# Q10. Kế hoạch trả góp (installments) có giá trị thanh toán trung bình cao nhất
q10_ans = payments.groupby('installments')['payment_value'].mean().idxmax()
print(f"Q10: Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất: {q10_ans} kỳ")

Q10: Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất: 6 kỳ
